## LangChain

LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs).

* It provides tools for connecting LLMs to external data sources, like databases or APIs, and for creating chains of actions.
* This allows developers to build sophisticated applications that go beyond simple text generation.
* LangChain facilitates tasks like document summarization, question answering, and creating agents that can interact with the real world.
* By offering modular components and abstractions, it accelerates LLM app development, making complex workflows more manageable.

For more information, please refer to https://www.langchain.com/

## Retrieval Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a technique that enhances the capabilities of large language models (LLMs) by grounding them in external knowledge sources. Instead of relying solely on their pre-trained data, RAG allows LLMs to retrieve relevant information from a database or knowledge base in real-time.

Here's how it works: when a user poses a question, the system first retrieves relevant documents or passages from the external source. This retrieved information is then combined with the user's query and fed into the LLM, which generates a response based on both the retrieved knowledge and its pre-existing understanding.

RAG improves accuracy, reduces hallucinations, and enables LLMs to answer questions about information they weren't originally trained on.

 This is particularly useful for applications requiring up-to-date or domain-specific knowledge. RAG effectively bridges the gap between static LLMs and dynamic, real-world data.

## RAG Using LangChain

We will build a Retreival Augmented Generation Application using the following steps that we have discussed in the lesson.
* Step 1: Document Loading
* Step 2: Splitting Text into Chunks
* Step 3: Storage Text as Vectorstore
* Step 4: Query and Retreival text
* Step 5: Output answer with retreival text and LLM Augmented Generation

Dataset: encyclopedia of medicine PDF file

![RAG Workflow](https://miro.medium.com/v2/resize:fit:1200/format:webp/1*-TPXZmeTpI9hezbZqA4NpA.png)

##Load Libraries

In [1]:
!pip install -U langchain langchain_community langchain_groq langchain_chroma langchain_huggingface langchain_text_splitters chromadb python-dotenv gdown pypdf sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.3/611.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Download the .env into the colab virtual drive

In [3]:
import gdown
url = 'https://drive.google.com/file/d/1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm/view?usp=drive_link'
output_path = '.env'
gdown.download(url, output_path, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1f-X_cbCcJG0JrJl2FsCNceuA6POKjnXm
To: /content/.env
100%|██████████| 71.0/71.0 [00:00<00:00, 164kB/s]


'.env'

In [4]:
from dotenv import load_dotenv
import os
# load .env file to environment
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

---
> NOTE: Please DO NOT use the Groq API Key outside of this workshop. Get a free key at https://console.groq.com/keys and store it in your `.env` file as `GROQ_API_KEY=...`
---

In [5]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.9)


#Test the LLM
print(llm.invoke([{'role':'user', 'content':'Which is the largest country by area in the world?'}]).content)

The largest country by area in the world is Russia, covering approximately 17.1 million square kilometers (6.6 million square miles). It spans across much of northern Eurasia and accounts for about 11% of the Earth's land area.


##Step 1: Document Loading

Create a directory name data. Copy the pdf document into the directory created. Filename: encyclopedia-of-medicine.pdf

In [6]:
!mkdir data

Read the pdf doucment into multiple pages of text.

In [7]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# Extract Data From the PDF File
def load_pdf_file(data):
    loader = DirectoryLoader(data, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

extracted_data = load_pdf_file(data='./data/')

/tmp/ipykernel_547/1923104849.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [8]:
print(type(extracted_data), len(extracted_data))   # Organize by the pages. Total pages: 68.

<class 'list'> 20


In [ ]:
extracted_data[2]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-11T10:00:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-11T10:00:44+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/VirtuWorks_Expanded_HR_Policy_SG.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2'}, page_content='1. Introduction\nThis HR Policy outlines the employment practices, responsibilities, and benefits for all employees\nand contractors of VirtuWorks Pte. Ltd., a remote-first company based in Singapore. It ensures\nconsistency, compliance with Singapore employment laws (Employment Act, CPF, PDPA), and\nsupports a positive and productive virtual workplace.\nScope: Applies to all full-time, part-time, interns, and contract employees working remotely for\nVirtuWorks Pte. Ltd.\nHR Contact: hr@virtworks.sg | Company Address: 100 Peck Seah Street, #08-14 PS100,\nSingapore 079333\

In [ ]:
extracted_data[-1]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-11T10:00:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-11T10:00:44+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/VirtuWorks_Expanded_HR_Policy_SG.pdf', 'total_pages': 16, 'page': 15, 'page_label': '16'}, page_content='15. Policy Acknowledgment & Compliance\n\x7f\nEmployees must acknowledge reading, understanding, and complying with this HR Policy.\n\x7f\nViolations may result in corrective or disciplinary action, up to termination.\n\x7f\nHR will maintain records of acknowledgment forms.')

##Step 2: Splitting Text into Chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)  # size by characters
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 35


In [ ]:
text_chunks[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-30T02:45:17+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-30T02:45:17+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/VirtuWorks_Job_Postings_Text.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='VirtuWorks Pte Ltd - Internal Job Postings (Text\n Version)\nVirtuWorks Pte Ltd - Internal Job Postings (Text Version)\n------------------------------------------------------------\nJob Title: Software Engineer (Full-Stack)\nDepartment: Product Development\nReports To: Lead Developer\nDescription: Design, develop, and maintain full-stack web applications using React, Node.js, and\nPython.\nKey Responsibilities: Develop APIs, manage CI/CD pipelines, conduct code reviews.')

In [ ]:
text_chunks[1]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-30T02:45:17+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-30T02:45:17+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/VirtuWorks_Job_Postings_Text.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Requirements: Degree in Computer Science, experience with JavaScript, REST APIs, and cloud\nplatforms.\n------------------------------------------------------------\nJob Title: AI/ML Engineer\nDepartment: Research & Innovation\nReports To: Head of AI Systems\nDescription: Develop and deploy AI/ML models for predictive analytics and generative AI\napplications.\nKey Responsibilities: Build LLM models, implement RAG pipelines, monitor and optimize\nperformance.')

In [ ]:
text_chunks[-1]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-10-11T10:00:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-10-11T10:00:44+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'data/VirtuWorks_Expanded_HR_Policy_SG.pdf', 'total_pages': 16, 'page': 15, 'page_label': '16'}, page_content='15. Policy Acknowledgment & Compliance\n\x7f\nEmployees must acknowledge reading, understanding, and complying with this HR Policy.\n\x7f\nViolations may result in corrective or disciplinary action, up to termination.\n\x7f\nHR will maintain records of acknowledgment forms.')

##Step 3: Storage Text as Vectorstore

Vector databases are specialized databases designed to efficiently store, manage, and query high-dimensional vector data.

These vectors represent data points in a multi-dimensional space, capturing the semantic meaning or features of items like text, images, or audio. Unlike traditional databases that rely on structured data and exact matches, vector databases excel at similarity searches.

![Vector Space](https://images.contentstack.io/v3/assets/blt7151619cb9560896/blt5c3b8fcafa132cf2/667daaeb82ce1d23f7312f32/lorbgyz9ffm8ui8jh-vector-database-search1.png)

> **Note:** Groq does not currently offer an embeddings API, so this notebook uses a free, local Hugging Face sentence-transformer model (`sentence-transformers/all-MiniLM-L6-v2`) for embeddings, while Groq is used for the fast LLM generation step.

In [10]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma.from_texts([t.page_content for t in text_chunks],
                             embeddings,
                             collection_name="meddoc",
                             persist_directory="./meddoc_db")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##Step 4: Query and Retreival text

In [11]:
question = "What my work hours?"
results = vectordb.similarity_search(question, k=3)
print(len(results))
print(results[0])

3
page_content='4. Working Hours & Flexibility

Flexible work schedules with core collaboration hours 10:00 AM to 3:00 PM SGT.

Employees are expected to manage schedules responsibly and attend required meetings.

Time-off requests must be submitted in advance via HR-approved tools.

Work-from-anywhere policy applies unless restricted for security reasons.

Managing multiple employment or side projects requires disclosure and approval.'


In [ ]:
question = "I am look for a Post in DevOp Engineer"
results = vectordb.similarity_search(question, k=5)
print(len(results))
print(results[0])

5
page_content='performance.
Requirements: Proficiency in Python, PyTorch, LangChain, LlamaIndex, and vector databases.
------------------------------------------------------------
Job Title: DevOps Engineer
Department: Infrastructure
Reports To: CTO
Description: Maintain cloud infrastructure and automate deployment processes for internal and
client systems.
Key Responsibilities: Manage CI/CD pipelines, configure Kubernetes, implement monitoring.'


In [14]:
# Specifying top k
retriever = vectordb.as_retriever(search_kwargs={ "k" : 10})
print(retriever.invoke("I am looking for Post in DevOp Engineer")[0])

page_content='VirtuWorks Pte Ltd - Internal Job Postings (Text
 Version)
VirtuWorks Pte Ltd - Internal Job Postings (Text Version)
------------------------------------------------------------
Job Title: Software Engineer (Full-Stack)
Department: Product Development
Reports To: Lead Developer
Description: Design, develop, and maintain full-stack web applications using React, Node.js, and
Python.
Key Responsibilities: Develop APIs, manage CI/CD pipelines, conduct code reviews.'


##Step 5: Output answer with retreival text and LLM Augmented Generation

Augmentation

In [15]:
from langchain_core.prompts import ChatPromptTemplate

TEMPLATE = """\
You are  VirtuWorks HR assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

Generation

Finally, we are going to create a RAG Chain. For that, we are going to use LCEL (LangChain Expression Language) Runnable function.



In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

setup_and_retrieval = RunnableParallel({"question": RunnablePassthrough(),
                                        "context": retriever })
output_parser = StrOutputParser()
retrieval_chain = setup_and_retrieval | rag_prompt | llm | output_parser
retrieval_chain.invoke( "What my work hours?")

In [ ]:
retrieval_chain.invoke( "What is the COVID?")


"I don't know."

In [ ]:
print(retrieval_chain.invoke( "I looking for internal work Posting"))


Here are the current internal job postings at VirtuWorks Pte Ltd:

1. **Software Engineer (Full-Stack)**
   - **Department:** Product Development
   - **Reports To:** Lead Developer
   - **Description:** Design, develop, and maintain full-stack web applications using React, Node.js, and Python.
   - **Key Responsibilities:** Develop APIs, manage CI/CD pipelines, conduct code reviews.
   - **Requirements:** Proficiency in Python, PyTorch, LangChain, LlamaIndex, and vector databases.

2. **DevOps Engineer**
   - **Department:** Infrastructure
   - **Reports To:** CTO
   - **Description:** Maintain cloud infrastructure and automate deployment processes for internal and client systems.
   - **Key Responsibilities:** Manage CI/CD pipelines, configure Kubernetes, implement monitoring.
   - **Requirements:** AWS, Terraform, Ansible, and Linux scripting experience.

3. **UI/UX Designer**
   - **Department:** Design & Experience
   - **Reports To:** Product Manager
   - **Description:** Create 

In [ ]:
print(retrieval_chain.invoke("What is the company Employment structure?"))

The employment structure at VirtuWorks Pte. Ltd. includes the following types of employment:

- **Employment Types**: Permanent, part-time, contractual, and freelance.
- **Probation Period**: Typically lasts 3–6 months, with confirmation based on satisfactory performance.
- **Employee Classification**: Employees are classified for CPF contributions, benefits, and leave entitlement.
- **Equal Opportunity Employment**: The company emphasizes no discrimination based on race, gender, religion, or other protected attributes.
